In [4]:
import time
import numpy as np, pandas as pd
from itertools import combinations
from sklearn.metrics import roc_auc_score

def paired_bootstrap_auc_fast(qa, qb, n_boot=1000, seed=0):
    m = qa[["qidx", "uid", "y_true", "p_late_mean"]].merge(
        qb[["qidx", "p_late_mean"]], on="qidx", suffixes=("_a", "_b"))
    assert len(m) == len(qa) == len(qb), "hai lần chạy phải có cùng tập câu hỏi"
    assert not m[["p_late_mean_a", "p_late_mean_b"]].isna().any().any(), "preds có NaN"
    y = m["y_true"].to_numpy().astype(np.float64)
    ucode, _ = pd.factorize(m["uid"])
    n_u = ucode.max() + 1

    def prep(p):
        _, inv = np.unique(p, return_inverse=True)
        return inv, inv.max() + 1
    ia, ga = prep(m["p_late_mean_a"].to_numpy())
    ib, gb = prep(m["p_late_mean_b"].to_numpy())

    def wauc(inv, g, w):
        wp = np.bincount(inv, weights=w * y, minlength=g)
        wn = np.bincount(inv, weights=w * (1 - y), minlength=g)
        cn = np.cumsum(wn) - wn
        return (wp * (cn + 0.5 * wn)).sum() / (wp.sum() * wn.sum())

    ones = np.ones(len(y))
    d0 = wauc(ib, gb, ones) - wauc(ia, ga, ones)
    rng = np.random.default_rng(seed)
    ds = np.empty(n_boot)
    for i in range(n_boot):
        counts = np.bincount(rng.integers(0, n_u, n_u), minlength=n_u)
        w = counts[ucode].astype(np.float64)
        ds[i] = wauc(ib, gb, w) - wauc(ia, ga, w)
    lo, hi = np.percentile(ds, [2.5, 97.5])
    return d0, lo, hi

runs = {
    "a": pd.read_parquet("./preds_a.parquet"),
    "b": pd.read_parquet("./preds_b.parquet"),
    "c": pd.read_parquet("./preds_c.parquet"),
}
print({k: len(v) for k, v in runs.items()})

# AUC từng mô hình
for k, q in runs.items():
    print(f"AUC_{k} = {roc_auc_score(q['y_true'], q['p_late_mean']):.4f}")

# Kiểm tra bản nhanh khớp sklearn (d0 phải bằng AUC_b − AUC_a)
d0_chk = paired_bootstrap_auc_fast(runs["a"], runs["b"], n_boot=5)[0]
d0_ref = (roc_auc_score(runs["b"]["y_true"], runs["b"]["p_late_mean"])
          - roc_auc_score(runs["a"]["y_true"], runs["a"]["p_late_mean"]))
print(f"check: fast={d0_chk:.6f}  sklearn={d0_ref:.6f}")

# So sánh từng cặp (delta = AUC_sau − AUC_trước)
rows = []
for x, y in combinations(runs, 2):
    t = time.time()
    d0, lo, hi = paired_bootstrap_auc_fast(runs[x], runs[y], n_boot=1000, seed=0)
    print(f"{y} − {x} xong sau {time.time()-t:.0f}s")
    rows.append({"so sánh": f"{y} − {x}", "ΔAUC": d0, "CI_low": lo, "CI_high": hi,
                 "khác biệt có ý nghĩa": (lo > 0) or (hi < 0)})

print(pd.DataFrame(rows).round(4).to_string(index=False))

{'a': 1099197, 'b': 1099197, 'c': 1099197}
AUC_a = 0.7838
AUC_b = 0.7841
AUC_c = 0.7891
check: fast=0.000289  sklearn=0.000289
b − a xong sau 56s
c − a xong sau 54s
c − b xong sau 52s
so sánh   ΔAUC  CI_low  CI_high  khác biệt có ý nghĩa
  b − a 0.0003  0.0001   0.0005                  True
  c − a 0.0053  0.0048   0.0057                  True
  c − b 0.0050  0.0046   0.0054                  True
